# BERT — Семинар 2 (Teacher)

**Тема:** BERT: fine-tuning и абляции  
**Формат:** семинар (90 минут)  
**Описание:** Продвинутые практики с BERT: fine-tuning под классификацию, абляции (заморозка слоёв, длина контекста, скорость/качество), эмбеддинги vs fine-tuning, анализ ошибок и интерпретация результатов.

## Цели занятия

- Освоить полный цикл fine-tuning BERT/DistilBERT для классификации текста через `Trainer`.
- Сравнить стратегии обучения: full fine-tuning vs заморозка энкодера и обучение только головы.
- Понять влияние ключевых гиперпараметров (learning rate, max_length, batch size) на метрики и стабильность.
- Сравнить инженерные альтернативы: «замороженный encoder + простая модель» vs fine-tuning.
- Провести анализ ошибок: confusion matrix, просмотр трудных примеров и качественная интерпретация.

## Подготовка окружения

Ниже — единый набор импортов и вспомогательных функций.  
Все последующие ячейки предполагают запуск **сверху вниз**.

In [ ]:
# Если вы запускаете в Colab/новом окружении, может понадобиться установка:
# !pip -q install transformers datasets evaluate scikit-learn

import os
import time
import random
import numpy as np
import torch

import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, List, Tuple

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Теория 1: fine-tuning BERT для классификации

Мы берём предобученную encoder-only модель и добавляем **классификационную голову**.  
Fine-tuning означает, что обновляются **и** веса энкодера, **и** веса головы.

Практически это реализуется через `AutoModelForSequenceClassification` и `Trainer`.

Ключевые гиперпараметры:
- `learning_rate`: слишком большой → нестабильность; слишком маленький → медленное обучение.
- `max_length`: влияет на качество (контекст) и скорость (квадратичная стоимость attention).
- `batch_size`: влияет на шум градиента и скорость.

## Практическая часть

Ниже — 5 задач. Мы будем использовать небольшой срез датасета, чтобы семинар был воспроизводим даже на CPU.

### Задача 1. Fine-tuning DistilBERT/BERT для бинарной классификации (sentiment)

**Постановка.**  
1) Загрузите датасет `rotten_tomatoes` (labels 0/1).  
2) Возьмите небольшой срез train/test.  
3) Обучите `AutoModelForSequenceClassification` 1 эпоху.  
4) Посчитайте accuracy и F1.

**Теоретический минимум.**  
- `Trainer` берёт на себя батчинг, оптимизацию и оценку.  
- При маленьком числе шагов метрики будут шумными — это нормально для учебного эксперимента.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
import evaluate

set_seed(42)

dataset = load_dataset("rotten_tomatoes")
train_raw = dataset["train"].shuffle(seed=42).select(range(800))
test_raw = dataset["test"].shuffle(seed=42).select(range(400))

model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tok = train_raw.map(preprocess, batched=True)
test_tok = test_raw.map(preprocess, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

args = TrainingArguments(
    output_dir="bert_seminar2_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
metrics = trainer.evaluate()
metrics

**Интерпретация результата.**  
- Для малого числа шагов значения метрик нестабильны.  
- Сравнивайте стратегии и абляции **в одном и том же протоколе** (одинаковые данные/эпохи).

**Ожидаемые результаты (ориентиры).**  
На небольшом срезе `rotten_tomatoes` после 1 эпохи обычно можно увидеть accuracy порядка ~0.75–0.85,  
но это зависит от устройства, seed и случайного поднабора.

### Задача 2. Абляция: «заморозить энкодер» vs «fine-tune всё»

**Постановка.**  
Проведите два обучения:
1) full fine-tuning (как в Задаче 1),  
2) заморозка всех слоёв энкодера и обучение только классификационной головы.

Сравните метрики и время обучения.

**Теоретический минимум.**  
Заморозка уменьшает число обучаемых параметров → быстрее и стабильнее на слабом железе,  
но может проигрывать по качеству, особенно если задача отличается от домена предобучения.

In [ ]:
def freeze_encoder(model):
    # DistilBERT: encoder лежит в model.distilbert
    for name, param in model.named_parameters():
        if name.startswith("distilbert."):
            param.requires_grad = False
    return model

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_one(model, lr=2e-5, tag="run"):
    args = TrainingArguments(
        output_dir=f"bert_seminar2_{tag}",
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=1,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    t0 = time.time()
    trainer.train()
    dt = time.time() - t0
    m = trainer.evaluate()
    return dt, m

# 1) Full fine-tuning
model_full = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
print("Trainable params (full):", count_trainable_params(model_full))
dt_full, m_full = train_one(model_full, tag="full")

# 2) Freeze encoder
model_frozen = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
model_frozen = freeze_encoder(model_frozen)
print("Trainable params (frozen):", count_trainable_params(model_frozen))
dt_frz, m_frz = train_one(model_frozen, tag="frozen")

print("\n=== Сравнение ===")
print("Full fine-tuning time (s):", round(dt_full, 2), "| metrics:", m_full)
print("Frozen encoder time (s):", round(dt_frz, 2), "| metrics:", m_frz)

**Интерпретация результата.**  
- Если качество при заморозке сильно падает, это ожидаемо: энкодер не адаптируется под задачу.  
- Если падение небольшое, то заморозка может быть хорошим компромиссом для продакшена/ограниченных ресурсов.

**Ожидаемые результаты (ориентиры).**
- Время при заморозке обычно заметно меньше (особенно на CPU).  
- Метрики при заморозке часто ниже, но иногда разница умеренная на простых задачах.

### Задача 3. Альтернатива fine-tuning: «замороженный encoder + простая модель» (CLS vs mean pooling)

**Постановка.**  
1) Зафиксируйте encoder (без обучения).  
2) Получите эмбеддинги текстов:
   - вариант A: `CLS`-вектор,
   - вариант B: mean pooling по токенам.
3) Обучите `LogisticRegression` и сравните качество.

**Теоретический минимум.**  
Это типичный инженерный baseline: быстро, дёшево, воспроизводимо.  
Важно: «сырой CLS» не всегда хорош для sentence embeddings, поэтому сравнение с mean pooling показательно.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoModel

enc = AutoModel.from_pretrained(model_id, output_hidden_states=False, return_dict=True).to(device)
enc.eval()

def embed_texts(texts: List[str], mode: str = "cls", batch_size: int = 32) -> np.ndarray:
    assert mode in ("cls", "mean")
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch_text = texts[i:i+batch_size]
        toks = tokenizer(batch_text, truncation=True, max_length=128, padding=True, return_tensors="pt")
        toks = {k: v.to(device) for k, v in toks.items()}
        with torch.no_grad():
            out = enc(**toks)
            Z = out.last_hidden_state  # (b, seq, hidden)
        if mode == "cls":
            vec = Z[:, 0, :]
        else:
            mask = toks["attention_mask"].unsqueeze(-1)
            vec = (Z * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_vecs.append(vec.detach().cpu().numpy())
    return np.vstack(all_vecs)

X_train_cls = embed_texts(train_raw["text"], mode="cls")
X_test_cls = embed_texts(test_raw["text"], mode="cls")
X_train_mean = embed_texts(train_raw["text"], mode="mean")
X_test_mean = embed_texts(test_raw["text"], mode="mean")

y_train = np.array(train_raw["label"])
y_test = np.array(test_raw["label"])

def fit_eval(Xtr, Xte, tag):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    return {
        "tag": tag,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    }

res_cls = fit_eval(X_train_cls, X_test_cls, "CLS + LogisticRegression")
res_mean = fit_eval(X_train_mean, X_test_mean, "MeanPool + LogisticRegression")

res_cls, res_mean

**Интерпретация результата.**  
- Часто mean pooling даёт более стабильный baseline, чем «сырой CLS».  
- Fine-tuning обычно выигрывает, но ценой времени/ресурсов и риска переобучения на малых данных.

**Ожидаемые результаты (ориентиры).**
- Baseline с эмбеддингами часто уступает fine-tuning, но может быть «достаточно хорошим».  
- Разница между CLS и mean pooling зависит от модели; на практике mean pooling часто выигрывает.

### Задача 4. Анализ ошибок: confusion matrix и «трудные» примеры

**Постановка.**  
1) Возьмите лучший из подходов (например, fine-tuned модель из Задачи 1) и получите предсказания на test.  
2) Постройте confusion matrix.  
3) Выведите 5–10 примеров с высокой уверенностью, но неверной классификацией.

**Теоретический минимум.**  
Анализ ошибок часто важнее +0.5% метрики: он показывает, *где* модель систематически ошибается.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Возьмём trainer из Задачи 1 (он содержит модель после обучения)
preds = trainer.predict(test_tok)
logits = preds.predictions
y_true = preds.label_ids
y_pred = np.argmax(logits, axis=-1)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["neg(0)", "pos(1)"])
disp.plot()
plt.title("Confusion matrix (test)")
plt.show()

# найдём ошибки с высокой уверенностью (softmax)
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
conf = probs.max(axis=-1)
wrong_idx = np.where(y_pred != y_true)[0]
# сортировка по уверенности
wrong_sorted = wrong_idx[np.argsort(-conf[wrong_idx])]

print("Топ-8 уверенных ошибок:")
for i in wrong_sorted[:8]:
    text = test_raw["text"][i]
    print("-"*80)
    print("text:", text[:300].replace("\n", " "))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "conf:", float(conf[i]))

**Интерпретация результата.**  
- Частые причины ошибок в sentiment: сарказм, отрицания, «смешанная» оценка, длинные отзывы, редкие выражения.  
- Высокая уверенность при ошибке — сигнал, что модель выучила ложные корреляции.

### Задача 5. Абляция: влияние `max_length` на скорость и качество

**Постановка.**  
Проведите короткий эксперимент: `max_length ∈ {64, 128, 256}`.  
Для каждого значения:
- заново токенизируйте данные,
- обучите модель 1 эпоху,
- замерьте время и метрики.

**Теоретический минимум.**  
Стоимость self-attention в Transformer растёт как \(O(L^2)\), где \(L\) — длина последовательности.

In [ ]:
def run_maxlen_experiment(max_len: int) -> Dict[str, float]:
    train_tok2 = train_raw.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_len), batched=True)
    test_tok2 = test_raw.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_len), batched=True)

    model2 = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    args2 = TrainingArguments(
        output_dir=f"bert_seminar2_maxlen_{max_len}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=1,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=200,
        report_to="none",
    )
    trainer2 = Trainer(
        model=model2,
        args=args2,
        train_dataset=train_tok2,
        eval_dataset=test_tok2,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    t0 = time.time()
    trainer2.train()
    dt = time.time() - t0
    m = trainer2.evaluate()
    return {
        "max_length": max_len,
        "time_sec": dt,
        "accuracy": m.get("eval_accuracy"),
        "f1": m.get("eval_f1")
    }

results = [run_maxlen_experiment(L) for L in [64, 128, 256]]
results

**Интерпретация результата.**  
- Обычно `max_length=64` быстрее, но может терять контекст.  
- `max_length=256` часто заметно медленнее, а прирост качества не гарантирован (зависит от задачи и данных).

**Ожидаемые результаты (ориентиры).**
- Время обучения растёт с `max_length` (часто нелинейно).  
- Качество может слегка расти до некоторого предела и затем насыщаться.

## Интерпретация результатов (сводный анализ)

1) Fine-tuning даёт сильный прирост качества, потому что энкодер **адаптируется** под задачу.  
2) Заморозка энкодера ускоряет обучение и может быть разумным компромиссом, но часто снижает качество.  
3) Baseline «эмбеддинги + простая модель» полезен как быстрый ориентир и иногда достаточно хорош на практике.  
4) Ошибки с высокой уверенностью показывают, где модель «уверенно ошибается» — это ключ для улучшений.  
5) `max_length` влияет не только на качество, но и на стоимость — это инженерный trade-off.

## Выводы

- Fine-tuning BERT — это стандартный путь к высоким метрикам на конкретной задаче.  
- В условиях ограниченных ресурсов разумно начинать с:
  1) baseline (эмбеддинги + простой классификатор),  
  2) затем заморозка части слоёв,  
  3) и только потом full fine-tuning.
- Абляции — обязательный инструмент: они превращают «магическое улучшение» в объяснимую причинность.

## Самостоятельные задачи (для закрепления)

1) Проведите абляцию по `learning_rate ∈ {1e-5, 2e-5, 5e-5}` при фиксированном `max_length`.  
2) Попробуйте модель `bert-base-uncased` вместо `distilbert-base-uncased` и сравните скорость/качество.  
3) Реализуйте частичную заморозку: разморозьте только последние 2 Transformer-блока и голову.  
4) Для ошибок из Задачи 4 сгруппируйте примеры по причинам (сарказм, отрицания, длинные отзывы и т.д.).

**Эталонные решения (кратко).**

1) Для LR-абляции достаточно обернуть обучение в функцию (как в Задаче 5) и менять `learning_rate`.  
   Часто слишком большой LR даёт скачки loss и ухудшение метрик.

2) `bert-base-uncased` обычно медленнее (больше параметров), иногда чуть лучше по метрикам.

3) Частичная заморозка:
- для DistilBERT можно заморозить первые N слоёв в `model.distilbert.transformer.layer[:N]`
  выставив `requires_grad=False`.
- затем обучать.

4) Кластеризация причин ошибок — вручную по 20–30 примерам; цель — найти паттерны и решить, нужна ли
   доразметка данных, доменная адаптация или изменение препроцессинга.